In [ ]:
import pandas as pd
import xarray as xr

ds = xr.open_dataset("../data/raw/reanalysis-era5-pressure-levels_2001-03_geopotential.nc", engine="netcdf4")
# print(ds)

try:
    ds_surface = xr.open_dataset("../data/raw/reanalysis-era5-single-levels_2001-03.nc", engine="netcdf4")
    print(ds_surface)
except Exception as e:
    print(f"Could not open file, error: {e}")


Hmm, the file format was unknown? Why did engine="netcdf4" work for the pressure-levels data but not the surface-level data?

My intuition tells me that sometimes servers will automatically zip files together without labelling them gz or zip. Let's see if that's the case here:

In [ ]:
import zipfile
import os

try:
    in_path = "../data/raw/reanalysis-era5-single-levels_2001-03.nc"
    out_path = "../data/raw/temp_single-levels_2001-03.nc"
    if zipfile.is_zipfile(in_path):
        print("is zip file")
        with zipfile.ZipFile(in_path, 'r') as zip_ref:
            zip_ref.extractall(out_path)
        search_pattern = os.path.join(out_path, "*.nc")
        ds = xr.open_mfdataset(search_pattern, engine="netcdf4", combine="by_coords")
        print(ds)
except Exception as e:
    print(f"Still couldn't open file, error: {e}")


Great, that works now.

Let's see what we can do with just this data.

In [ ]:
import matplotlib.pyplot as plt

ds.sel(latitude=41.0, longitude=-78.0).plot.scatter(x="valid_time", y="t2m")
# ds.where(ds["tp"] > 0).plot.scatter(x="valid_time", y="tp")
tmax = ds.t2m.sel(latitude=41.0, longitude=-78.0).resample(valid_time="1D").max()
# print(tmax)
tmax.plot(marker='o', linestyle='', markersize=3)
plt.title("Daily Max Temperature")
df = ds.to_dataframe().reset_index()
# print(df)

Generating a basic ML model to test on the dataset:

In [ ]:
import numpy as np

# formatting data for ML model use

# print(df)
y = df.tp
for i in range(len(y)):
    if y[i] > 0:
        y[i] = True
    else: y[i] = False
print(y)

assert(len(df) == len(y))

n = df.shape[0]
print(n)
d = df.shape[1] - 1
print(d)
X = np.zeros((n, d))
for i in range(d):
    mapping = {name: j for j, name in enumerate(sorted(set(df[:, i+1])))}
    for j in range(n):
        X[j, i] = mapping[df[j, i+1]]
print(X)

In [ ]:
import sklearn
import pytest
import sklearn.neighbors

class Classifier(object):
    def fit(self, X, y):
        raise NotImplementedError()
    
    def predict(self, X):
        raise NotImplementedError()

def create_kNeighbors():
    clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors=5)
    clf.fit
